## Entregable 2

Somos una empresa farmacéutica y queremos desarrollar nuevos fármacos y experimentos. Las experimentaciones en el sector farmacéutico son públicas y en este ejercicio se recoge una pequeña muestra de 5 referencias.
El ejercicio consiste en crear una tabla en la que cada fila sea la información de cada documento, y cada columna sea una variable de interés. Puede ocurrir que la variable de interés no aparezca en todos los documentos, en cuyo caso se pide imputar un nulo.

Para llevar a cabo este ejercicio se propone utilizar una LLM que se puede descargar en local a través de:
* https://huggingface.co/models?other=LLM

In [1]:
from pathlib import Path

# Project root directory
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# Create output directory if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Retrieve PDF files
pdf_files = sorted(RAW_DATA_DIR.glob("*.pdf"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Number of PDF files: {len(pdf_files)}")

for pdf in pdf_files:
    print(f"- {pdf.name}")

Project root: c:\Users\lydia\Desktop\LLM-Based-Pharma-Data-Extraction
Number of PDF files: 6
- fpsyg-08-00308.pdf
- fpsyg-09-01240.pdf
- fpsyg-11-01354.pdf
- fpsyg-12-639236.pdf
- fpsyt-14-1301143.pdf
- fpubh-10-1045777.pdf


In [2]:
import fitz

def extract_pdf_text(pdf_path):
    """
    Extract text from all pages of a PDF document.
    """
    pages = []

    with fitz.open(pdf_path) as doc:
        for page in doc:
            pages.append(page.get_text(sort=True))

    return "\n".join(pages)


documents = {}

for pdf_path in pdf_files:
    text = extract_pdf_text(pdf_path)

    documents[pdf_path.name] = text

    print(
        f"{pdf_path.name}: "
        f"{len(text):,} characters extracted"
    )

c:\Users\lydia\Desktop\LLM-Based-Pharma-Data-Extraction\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


fpsyg-08-00308.pdf: 109,125 characters extracted
fpsyg-09-01240.pdf: 142,856 characters extracted
fpsyg-11-01354.pdf: 73,302 characters extracted
fpsyg-12-639236.pdf: 105,142 characters extracted
fpsyt-14-1301143.pdf: 58,456 characters extracted
fpubh-10-1045777.pdf: 64,731 characters extracted


In [3]:
import json
import re
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16
)

model.eval()

print("LLM loaded successfully!")
print(f"GPU: {torch.cuda.get_device_name(0)}")

W1107 10:22:33.709000 23324 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1107 10:22:38.650000 23324 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM loaded successfully!
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
EXTRACTION_FIELDS = [
    "title",
    "authors",
    "publication_year",
    "doi",
    "research_type",
    "research_objective",
    "methodology",
    "sample_size",
    "intervention",
    "control_group",
    "main_findings",
    "limitations",
    "conclusions"
]

In [5]:
SYSTEM_PROMPT = """
You are an expert scientific information extraction assistant.

Extract structured information from scientific research papers.

Rules:
1. Extract information only from the provided document.
2. Do not invent or assume missing information.
3. Return null when information is missing or not applicable.
4. Return ONLY a valid JSON object.
5. Use exactly the requested keys.
6. Do not confuse cited studies with the original research.
7. For theoretical papers, experimental variables should be null
   unless original experiments are reported.
"""

def create_prompt(document_text):

    schema = {field: None for field in EXTRACTION_FIELDS}

    return f"""
Extract the following variables from this scientific paper:

{json.dumps(schema, indent=2)}

Definitions:
- title: Full paper title.
- authors: Authors of the paper.
- publication_year: Publication year.
- doi: Digital Object Identifier.
- research_type: Type of research.
- research_objective: Main research objective.
- methodology: Methods used.
- sample_size: Number of participants in original experiments.
- intervention: Intervention studied.
- control_group: Control condition or group.
- main_findings: Main findings.
- limitations: Explicitly reported limitations.
- conclusions: Main conclusions.

Return valid JSON only.

DOCUMENT:
{document_text}
"""

In [6]:
MAX_INPUT_TOKENS = 3000
MAX_NEW_TOKENS = 1000

def extract_information(document_text):

    # Reserve space for instructions and JSON schema
    document_tokens = tokenizer.encode(
        document_text,
        add_special_tokens=False
    )

    document_tokens = document_tokens[:2000]

    truncated_document = tokenizer.decode(
        document_tokens,
        skip_special_tokens=True
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": create_prompt(truncated_document)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    if inputs.shape[-1] > MAX_INPUT_TOKENS:
        raise ValueError(
            "Prompt exceeds token limit. Reduce document tokens."
        )

    with torch.inference_mode():

        outputs = model.generate(
            inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return response

In [7]:
def parse_response(response):

    try:
        match = re.search(r"\{.*\}", response, re.DOTALL)

        if not match:
            raise ValueError("No JSON found")

        data = json.loads(match.group())

        return {
            field: data.get(field)
            for field in EXTRACTION_FIELDS
        }

    except (json.JSONDecodeError, ValueError) as error:
        print(f"Parsing error: {error}")
        return None

In [9]:
first_filename = list(documents.keys())[0]

print(f"Processing: {first_filename}")

response = extract_information(documents[first_filename])

print(response)

Processing: fpsyg-08-00308.pdf


AttributeError: 

In [ ]:
results = []
failed_documents = []

for filename, text in documents.items():

    print(f"Processing: {filename}")

    try:
        response = extract_information(text)
        data = parse_response(response)

        if data is None:
            failed_documents.append(filename)
            continue

        data["source_file"] = filename
        results.append(data)

        print("Completed!")

    except Exception as error:
        print(f"Error: {error}")
        failed_documents.append(filename)

print(f"\nSuccessfully processed: {len(results)}")
print(f"Failed documents: {len(failed_documents)}")

In [ ]:
df = pd.DataFrame(results)

column_order = ["source_file"] + EXTRACTION_FIELDS

df = df.reindex(columns=column_order)

display(df)

In [ ]:
csv_path = PROCESSED_DATA_DIR / "pharma_research_dataset.csv"
json_path = PROCESSED_DATA_DIR / "pharma_research_dataset.json"

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

df.to_json(
    json_path,
    orient="records",
    indent=4,
    force_ascii=False
)

print(f"CSV exported to: {csv_path}")
print(f"JSON exported to: {json_path}")